# データサイエンス特論 第12回 演習課題
青山学院大学大学院 理工学研究科 理工学専攻 知能情報コース 修士1年 35626302 森下剛

In [239]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
import numpy as np

## Lasso・Ridge回帰におけるハイパーパラメータの探索

#### 演習問題1の回答

In [240]:
# 糖尿病データセットの読み込み
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

# 全体の20%をテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

# 重回帰モデルを学習
lr = LinearRegression()
lr.fit(X_train, y_train)

# 予測と評価
y_pred = lr.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE = {rmse}")
print(f"MAE  = {mae}")

RMSE = 54.70449002870804
MAE  = 41.97492114949365


#### 演習問題2の回答

In [241]:
gs = GridSearchCV(Lasso(), {"alpha": [0.8, 1.0, 1.2]}, cv=10, scoring="neg_root_mean_squared_error")
gs.fit(X_train, y_train)
rmse = root_mean_squared_error(y_test, gs.predict(X_test))
print(f"alpha = {gs.best_params_['alpha']} : RMSE = {rmse}")

alpha = 0.8 : RMSE = 58.54280490441743


#### 演習問題3の回答

In [242]:
alphas = np.round(np.arange(0.1, 10.1, 0.1), 1)

In [243]:
gs = GridSearchCV(Lasso(), {"alpha": alphas}, cv=10, scoring="neg_root_mean_squared_error")
gs.fit(X_train, y_train)
rmse = root_mean_squared_error(y_test, gs.predict(X_test))
print(f"alpha = {gs.best_params_['alpha']} : RMSE = {rmse}")

alpha = 0.1 : RMSE = 55.0659448252393


In [244]:
alphas = np.round(np.arange(0.01, 2.01, 0.01), 2)

In [245]:
gs = GridSearchCV(Lasso(), {"alpha": alphas}, cv=10, scoring="neg_root_mean_squared_error")
gs.fit(X_train, y_train)
rmse = root_mean_squared_error(y_test, gs.predict(X_test))
print(f"alpha = {gs.best_params_['alpha']} : RMSE = {rmse}")

alpha = 0.06 : RMSE = 54.997124653109246


In [246]:
alphas = np.round(np.arange(0.051, 0.071, 0.001), 3)

In [247]:
gs = GridSearchCV(Lasso(), {"alpha": alphas}, cv=10, scoring="neg_root_mean_squared_error")
gs.fit(X_train, y_train)
rmse = root_mean_squared_error(y_test, gs.predict(X_test))
print(f"alpha = {gs.best_params_['alpha']} : RMSE = {rmse}")

alpha = 0.054 : RMSE = 54.99372095570399


ループをうまく作れば、計算時間を削減しつつ最良のパラメータを探索できそう。
調べたところ「反復的グリッド細分化」という手法がまさにこれに相当する。

#### 演習問題4の回答

In [248]:
gs = GridSearchCV(Ridge(), {"alpha": [0.8, 1.0, 1.2]}, cv=10, scoring="neg_root_mean_squared_error")
gs.fit(X_train, y_train)
rmse = root_mean_squared_error(y_test, gs.predict(X_test))
print(f"alpha = {gs.best_params_['alpha']} : RMSE = {rmse}")

alpha = 0.8 : RMSE = 56.619974420282034


Ridge回帰は係数の二乗値の総和を用いているので微分可能である。
よって、ハイパラに0を指定できる。

In [249]:
alphas = np.round(np.arange(0.0, 10.0, 0.1), 1)

In [250]:
gs = GridSearchCV(Ridge(), {"alpha": alphas}, cv=10, scoring="neg_root_mean_squared_error")
gs.fit(X_train, y_train)
rmse = root_mean_squared_error(y_test, gs.predict(X_test))
print(f"alpha = {gs.best_params_['alpha']} : RMSE = {rmse}")

alpha = 0.1 : RMSE = 54.98088640342304


In [251]:
depth = 4
low, high, step = 0.1, 10.0, 0.1

In [252]:
for i in range(depth):
    alphas = np.round(np.arange(low, high + step, step), 10)

    gs = GridSearchCV(Ridge(), {"alpha": alphas}, cv=10, scoring="neg_root_mean_squared_error")
    gs.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_test, gs.predict(X_test))
    best = gs.best_params_["alpha"]
    print(f"alpha = {best} : RMSE = {rmse}")

    # 更新
    low, high = best - step, best + step
    step = step / 10

alpha = 0.1 : RMSE = 54.98088640342304
alpha = 0.08 : RMSE = 54.988161876956774
alpha = 0.082 : RMSE = 54.987131156617096
alpha = 0.0818 : RMSE = 54.98723133198961


時間が余ったのでループ処理を実装してみた。